In [15]:
# Mounting git repo from Google Drive to this Colab session
from google.colab import drive, userdata
import os

# 1. Mount Google Drive for persistent storage
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Colab/

# 2. Fetch credentials from Colab Secrets
username = userdata.get('GH_USER')
token = userdata.get('GH_TOKEN')
repo_name = "NWC_W1"

# 3. Clone repo if missing (*should not happen)
if not os.path.exists(repo_name):
    repo_url = f"https://{token}@github.com/{username}/{repo_name}.git"
    !git clone {repo_url}
    print("✅ Repository cloned to Drive.")
else:
    print("📂 Repository already exists in Drive.")

# 4. Move into repo directory
%cd {repo_name}

# 5. Global Git Config (One-time per session)
!git config --global user.email "gsi.andras@gmail.com"
!git config --global user.name "kbandesz"



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab
📂 Repository already exists in Drive.
/content/drive/MyDrive/Colab/NWC_W1


# Workshop 1 – Standard Nowcasting Approaches: India Case Study
---

### Overview
This notebook demonstrates a workflow for nowcasting Indian Quarterly Real GDP using high-frequency monthly indicators. We will cover two primary methodologies:

1.  **Bridge Equations:** Forecasting monthly indicators to the end of the quarter, aggregating them, and running a standard quarterly regression.
2.  **MIDAS (Mixed Data Sampling):** Regressing low-frequency GDP directly on high-frequency monthly lags using:
    * **U-MIDAS:** Unrestricted skip-sampling.
    * **Restricted MIDAS:** Almon Polynomial Distributed Lags.

### Required files (downloaded automatically!)
* **Data:** `India.xlsx`.
* **X-13ARIMA-SEATS:** The `x13as_ascii` binary is required for seasonal adjustment.

## 0. Notes on India's Economy

India is bounded by the Indian Ocean on the south and the Arabian Sea on the southwest. It shares land borders with Pakistan, China, Nepal, Bhutan, Bangladesh, and Myanmar. In 2020, India was ranked as the 6th largest economy by GDP (in current US$). Before the COVID-19 pandemic, India was among the fastest-growing economies in the world.

According to the IMF Executive Board Article IV consultation with India in October 2021, India experienced two COVID-19 waves. The first caused an unprecedented contraction in real GDP of 7.3 percent in the fiscal year 2021. The second resulted in a smaller and shorter fall in activity. The combined impact of pandemic-related support and the contraction in economic activity led to a widening of the fiscal deficit to 12.8 percent of GDP.

<p align="center">
  <img src="https://drive.google.com/uc?id=18deMBjOr_b8lLHla39ec9_0g1axS0FBE" alt="India Real GDP Growth Rate" width="80%">
</p>

The economic outlook remains unclear given pandemic-related uncertainties. Medium-term growth will likely be impacted by adverse shocks in investment, human capital, and other growth factors. However, more efficient vaccination and better therapeutics should reduce the negative long-term impact of the pandemic.

India's Gross Domestic Product is highly dependent on private final consumption expenditures, accounting for more than 50 percent of the real GDP in 2020. Food and nonalcoholic beverages have the most weight in private consumption, accounting for 27 percent of real private consumption in 2019. Transport has the second highest weight, accounting for 18 percent of real private consumption in 2020.

<p align="center">
  <img src="https://drive.google.com/uc?id=1QcWaasiBA6_pFVC4tS08r_YAn1H7P9dr" alt="India Real GDP Decomposition" width="80%">
</p>

<p align="center">
  <img src="https://drive.google.com/uc?id=13DLWufhHIZ2ArZU731iyxx3lf-2AZVQn" alt="India Petroleum Consumption Breakdown" width="80%">
</p>

India's three largest trade partners (ranked from largest to smallest) are the United States, United Arab Emirates, and China. Consumer goods take the most weight among export products, accounting for 45 percent of exports. Among the import products, raw materials are at the top, accounting for 33 percent.

<p align="center">
  <img src="https://drive.google.com/uc?id=1fsO9q2QmB9O4BeRQ2fHpqVc3njQnOetP" alt="India Trade Partners 2019" width="80%">
</p>

Since the pandemic, India has experienced significant drops in the Purchasing Manager Index (PMI). The PMI dropped to a historical low of 7.1 in April 2020, reflecting the impact of 21 days of national lockdown that began on March 24th. However, the PMI recovered gradually after the easing of the lockdown, exceeding the neutral threshold (i.e., 50.0) in September. In November 2021, the PMI reached a ten-month high of 59.2, reflecting a solid expansion in the manufacturing sector.

<p align="center">
  <img src="https://drive.google.com/uc?id=1_q2P3HRmCIFPkRhQxalF1xT1SW6POPEr" alt="India PMI Composite Output" width="80%">
</p>

## 1. Setup & Installation
Run this cell first to install necessary libraries, download the Excel data file, and configure the X-13 seasonal adjustment tool.

In [ ]:
# 1. INSTALL LIBRARIES
print("Installing libraries...")
!pip install -q pmdarima statsmodels openpyxl gdown

# 2. IMPORTS
import os
import sys
import gdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pmdarima as pm
import re

# 3. Downloading necessary files (data + X-13 binary)
zip_id = "1JcmITiYoKHYQ8JaQ8gkeZO7Clq4fBIme" # from Google Drive share link
zip_filename = "W1_material.zip"

print("Downloading workshop materials...")
# Construct the download URL
url = f'https://drive.google.com/uc?id={zip_id}'
# Download (quiet=False shows the progress bar)
gdown.download(url, zip_filename, quiet=False)
# Unzip the files and delete zip file
!unzip -oq $zip_filename
!rm -f $zip_filename
print("Workshop files extracted.")

# 4. SETUP X-13ARIMA-SEATS (LINUX/COLAB)
print("Configuring X-13...")
try:
  # Set path for statsmodels (this assumes 'x13as_ascii' was uploaded to Colab).
  x13_binary = 'x13as_ascii'
  X13_PATH = os.path.join(os.getcwd(), x13_binary)
  # Ensure the binary has execute permissions
  !chmod +x "{X13_PATH}"
except Exception as e:
  print(f"X-13 Setup Failed: {e}.")

# 5. PLOTTING DEFAULTS
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("✅ Setup Complete.")

Installing libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 38.6 MB/s eta 0:00:00


Downloading...
From: https://drive.google.com/uc?id=1JcmITiYoKHYQ8JaQ8gkeZO7Clq4fBIme
To: /content/W1_material.zip
100%|██████████| 2.27M/2.27M [00:00<00:00, 184MB/s]


Workshop files extracted.
Configuring X-13...
✅ Setup Complete.


## 2. Data Loading & Structure Setup
The helper function below handles the specific formatting of `India.xlsx` (e.g., transposing data, fixing date headers). We load two dataframes:
1.  **Monthly Data (`df_m`):** High-frequency indicators (PMI, Motor Spirit Consumption).
2.  **Quarterly Data (`df_q`):** Target variable (India Real GDP), and other quarterly controls (US/CN Real GDP).

After loading the data, we extend the date index to include the Nowcast target period (up to June 2023).

In [ ]:
# Helper function to clean the specific 'India.xlsx' file.
# You do not need to worry about this data cleaning logic,
#  just understand that it converts the Excel sheets into clean Time Series.)
def import_india_data(file_path, sheet_name, freq):
    """
    Reads India.xlsx, handles date formatting (e.g. 199901 -> Date),
    and sets the index frequency.
    """
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=0)
    except FileNotFoundError:
        print(f"Error: {file_path} not found.")
        return pd.DataFrame()

    # Data starts at Column P (Index 15) in the provided Excel
    # Series IDs are in Column A (Index 0)
    df.set_index(df.columns[0], inplace=True)
    data_part = df.iloc[:, 14:] # Adjusting for 0-index, checking cols P onwards

    # Transpose: Dates become rows, Variables become columns
    df_t = data_part.T

    # Clean Date Index
    dates = []
    valid_rows = []
    for idx, row in df_t.iterrows():
        # Handle headers like 199901.0 (float) vs "199901" (str)
        d_str = re.sub(r'[^0-9]', '', str(idx))
        try:
            if freq == 'M' and len(d_str) == 6:
                # YYYYMM -> Date (Month Start)
                dates.append(pd.to_datetime(d_str, format='%Y%m'))
                valid_rows.append(idx)
            elif freq == 'Q' and len(d_str) == 5:
                # YYYYQ -> Date (Quarter End)
                y = int(d_str[:4])
                q = int(d_str[4:])
                # Create timestamp for END of quarter
                dates.append(pd.Timestamp(year=y, month=q*3, day=1) + pd.offsets.MonthEnd(1))
                valid_rows.append(idx)
        except ValueError:
            continue

    df_final = df_t.loc[valid_rows].copy()
    df_final.index = pd.DatetimeIndex(dates)
    df_final.index.name = 'Date'

    # Convert all columns to numeric
    return df_final.apply(pd.to_numeric, errors='coerce')

# --- Load Historical Data ---
print("Loading Data...")
df_m = import_india_data('India.xlsx', 'Monthly', 'M')
df_q = import_india_data('India.xlsx', 'Quarterly', 'Q')

df_m = df_m[['pmi_new','consu_motor']] # we will only work with these two series

# Rename variables
df_q.rename(columns={'rgdp_sa': 'rgdp'}, inplace=True)

# We extend dataframes to cover the Nowcast target: 2023Q2
target_end_m = pd.Timestamp('2023-06-01')
target_end_q = pd.Timestamp('2023-06-30')

print(f"\nExtending 'Monthly' index to {target_end_m.date()}...")
full_idx_m = pd.date_range(start=df_m.index.min(), end=target_end_m, freq='MS')
df_m = df_m.reindex(full_idx_m)

print(f"Extending 'Quarterly' index to {target_end_q.date()}...")
full_idx_q = pd.date_range(start=df_q.index.min(), end=target_end_q, freq='QE')
df_q = df_q.reindex(full_idx_q)

print("Structure Set. Future rows exist (as NaNs) ready to be filled.")

# Quick Plot
fig, ax = plt.subplots(2, 2, figsize=(10, 6))
print("\nQuarterly Data:")
df_q['rgdp'].plot(ax=ax[0, 0], title='India: Real GDP (Quarterly)')
df_q['rgdp_us'].plot(ax=ax[0, 1], title='US: Real GDP (Quarterly)')
df_q['rgdp_cn'].plot(ax=ax[1, 0], title='China: Real GDP (Quarterly)')
ax[1, 1].set_visible(False)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
print("\nMonthly Data:")
df_m['pmi_new'].plot(ax=ax[0], title='India: PMI (Monthly)')
df_m['consu_motor'].plot(ax=ax[1], title='India: Motor Spirit Consumption (Monthly)', color='orange')
plt.tight_layout()
plt.show()

# Quick table
print("\nQuarterly Data:")
print(df_q)
print("\nMonthly Data:")
print(df_m)


## 3. Data Transformation

We do some basic data transformations to create the variables for our nowcasting models.

### 3.1 Seasonal Adjustment (X-13)
Economic time series (like Motor Spirit consumption) often show strong seasonal patterns. We use **X-13ARIMA-SEATS** to remove these components. (Note: The PMI series is already seasonally adjusted by the data provider.)

In [ ]:
from statsmodels.tsa.x13 import x13_arima_analysis

# Check for target variable
target_var = 'consu_motor'

if target_var in df_m.columns:
    print(f"Performing Seasonal Adjustment on {target_var}...")
    try:
        # We use X-13 with outlier detection enabled
        sa_res = x13_arima_analysis(
            df_m[target_var].dropna(),
            x12path=X13_PATH,
            prefer_x13=True,
            outlier=True  # Detect outliers (e.g. Covid)
        )
        # Save Adjusted Series
        df_m[f'{target_var}_sa'] = sa_res.seasadj
        print("Success.")

        # Plot Comparison
        plt.figure(figsize=(10,4))
        plt.plot(df_m.index, df_m[target_var], label='Original', alpha=0.5)
        plt.plot(df_m.index, df_m[f'{target_var}_sa'], label='Seasonally Adjusted', linewidth=2)
        plt.title(f'{target_var}: Original vs SA')
        plt.legend()
        plt.show()

    except Exception as e:
        print(f"X-13 Failed: {e}. Is the binary path correct?")
else:
    print(f"Variable {target_var} not found.")

### 3.2 Growth Rates & Crisis Dummy
We convert levels to **Log-Differences (Growth Rates)** to ensure stationarity. We also create a **CRISIS** dummy to capture the negative effects of the GFC `(2009Q1)`, the 2016 demonetization `(2016Q4)`, and 2 COVID-19 waves `(2020Q2 and 2021Q2)`.

In [ ]:
# 1. Calculate Log-Diff Growth Rates
# Quarterly GDP Growth
df_q['dlog_rgdp'] = np.log(df_q['rgdp']).diff()

# External Factors (US & China GDP)
df_q['dlog_us'] = np.log(df_q['rgdp_us']).diff()
df_q['dlog_cn'] = np.log(df_q['rgdp_cn']).diff()

# 2. Create Crisis Dummy
# Dates: 2009Q1 (GFC), 2016Q4 (Demonetization), 2020Q2 & 2021Q2 (Covid)
df_q['CRISIS'] = 0
crisis_dates = ['2009-03-31', '2016-12-31', '2020-06-30', '2021-06-30']

for d in crisis_dates:
    if d in df_q.index:
        df_q.loc[d, 'CRISIS'] = 1

# 3. Plot QoQ growth and CRISIS episodes
fig, ax1 = plt.subplots(figsize=(10, 5))

# a. Plot GDP Growth (Primary Axis)
ax1.plot(df_q.index, df_q['dlog_rgdp'], color='navy', linewidth=2, label='GDP Growth')
ax1.set_ylabel('Log Growth Rate')
ax1.axhline(0, color='black', linewidth=0.5)

# 2. Plot Crisis Dummy (Secondary Axis)
ax2 = ax1.twinx()
ax2.bar(df_q.index, df_q['CRISIS'],
        color='grey', alpha=0.3,       # Make it look like shading
        width=-90, align='edge',       # Draw backwards from Q-End (approx 90 days)
        label='Crisis')

# Hide the secondary axis details
#ax2.axis('off')
ax2.set_ylim(0, 1) # Ensure bars touch the top

# Final Formatting
ax1.set_title('Indian Real GDP Growth & Crisis Episodes')
ax1.legend(loc='upper left')
plt.show()

## 4. The Bridge Model

**Thought experiment:** We are in June 2023. We have GDP data up to 2023Q1 and monthly data up to May.

**Goal:** Nowcast GDP for **2023Q2**.

We face a fundamental frequency mismatch: GDP is **Quarterly** (released every 3 months), but our indicators (PMI, Motor Spirit Consumption) are **Monthly**.

The **Bridge Equation** solves this in three logical steps:

1.  **Forecast Monthly Indicators:** Fill any missing months in the current quarter using a simple univariate model (ARIMA).
2.  **Aggregate:** Convert the full quarter of monthly data into a single quarterly value (Average or Sum).
3.  **Regress:** Run a standard "Quarterly on Quarterly" regression to predict GDP.



### 4.1 Fill monthly indicators with ARIMA forecast

In [ ]:
def fill_missing_values(series):
    """Forecasts NaNs at the end of a series using AutoARIMA."""
    # Split into 'known' (train) and 'unknown' (to forecast)
    known = series.dropna()
    last_valid_date = known.index[-1]

    # Identify how many steps we need to fill (rows in the series that are NaN)
    # Since 'series' comes from our extended dataframe, it HAS the future dates.
    full_range = series.loc[last_valid_date:].iloc[1:] # steps AFTER last valid
    steps = len(full_range)

    if steps == 0:
        return series

    print(f"Forecasting {steps} step(s) for {series.name}...")

    model = pm.auto_arima(known, seasonal=False, error_action='ignore', suppress_warnings=True)
    print(f"\nBest-fitting ARIMA Model for {series.name}:\n{model.summary()}\n")
    forecast = model.predict(n_periods=steps)

    # Fill the NaNs
    filled = series.copy()
    filled.loc[full_range.index] = forecast
    return filled

# Create Extended Series
df_m['pmi_ext'] = fill_missing_values(df_m['pmi_new'])
df_m['consu_ext'] = fill_missing_values(df_m['consu_motor_sa'])

# Verify
print("\nTail of Motor Spirit Consumption:")
print(df_m['consu_ext'].tail(3))

print("\nTail of PMI:")
print(df_m['pmi_ext'].dropna().tail(3))

### 4.2 Aggregate to quarterly

In [ ]:
# Assign to Quarterly DF (Alignment matches by index automatically)
df_q['pmi_q'] = df_m['pmi_ext'].resample('QE').mean()
df_q['consu_q'] = df_m['consu_ext'].resample('QE').sum()

# Recalculate Transforms (to ensure lags/diffs cover the new period)
df_q['dlog_consu'] = np.log(df_q['consu_q']).diff()
df_q['d_pmi'] = df_q['pmi_q'].diff()

# - Lags of GDP growth controls
for i in range(1, 4):
    df_q[f'dlog_rgdp_{i}'] = df_q['dlog_rgdp'].shift(i)

df_q['dlog_us_1'] = df_q['dlog_us'].shift(1)
df_q['dlog_cn_1'] = df_q['dlog_cn'].shift(1)

# Ensure CRISIS dummy is 0 for the future period (NaN -> 0)
df_q['CRISIS'] = df_q['CRISIS'].fillna(0)

print("Aggregation Complete. Variables ready for Bridge regression.")

### 4.3 Estimate Bridge regression and forecast

In [ ]:
# Set estimation sample and equation
est_mask = (df_q.index >= '2000-03-31') & (df_q.index <= '2023-03-31')

formula = "dlog_rgdp ~ dlog_rgdp_1 + dlog_rgdp_2 + dlog_rgdp_3 + dlog_consu + d_pmi + dlog_us_1 + dlog_cn_1 + CRISIS"

model_bridge = smf.ols(formula, data=df_q.loc[est_mask]).fit()
print("Bridge Equation - Estimation Results")
print(model_bridge.summary())

# --- NOWCAST (GROWTH & LEVELS) ---
target_row = df_q.loc['2023-06-30': '2023-06-30']
nowcast_growth = model_bridge.predict(target_row).values[0]

# Level Calculation: Level_t = Level_{t-1} * exp(Growth_t)
last_actual_level = df_q.loc['2023-03-31', 'rgdp']
nowcast_level = last_actual_level * np.exp(nowcast_growth)

print(f"\nBRIDGE NOWCAST 2023Q2:")
print(f"Growth: {nowcast_growth:.4f} (Log Diff)")
print(f"Level:  {nowcast_level:.2f} (INR Billions)")

# --- PLOT IN-SAMPLE FIT (LEVELS) ---
eval_mask = (df_q.index >= '2018-03-31') & (df_q.index <= '2023-03-31')
eval_data = df_q.loc[eval_mask]
fitted_growth = model_bridge.predict(eval_data)

# Reconstruct Levels
prev_levels = df_q['rgdp'].shift(1).loc[eval_mask]
fitted_levels = prev_levels * np.exp(fitted_growth)

plt.figure(figsize=(10, 5))
plt.plot(eval_data.index, eval_data['rgdp'], label='Actual', marker='o')
plt.plot(eval_data.index, fitted_levels, label='Bridge Fitted', linestyle='--')
plt.plot(target_row.index, [nowcast_level], label='Nowcast', marker='*', markersize=15)
plt.title('Bridge Model: Actual vs Fitted Levels')
plt.legend()
plt.show()

## 5. MIDAS Models (Mixed Data Sampling)

### The Concept

Bridge equations lose information by averaging the months. A strong shock in **Month 3** (late quarter) might impact GDP differently than a shock in **Month 1** (early quarter).

**MIDAS** solves this by running the regression directly on the monthly data, without aggregating first.

### The Equation
The regression relates Quarterly GDP ($y_t$) to a weighted sum of the Monthly Indicators ($x$):

$$y_t = \beta_0 + \beta_1 \sum_{k=0}^{K} w(k) x_{tm - k} + \epsilon_t$$

**Breakdown:**
* **$t$**: The Quarter (e.g., 2023Q2).
* **$tm$**: The last month of that quarter (June 2023).
* **$k$**: The high-frequency lag (0 = June, 1 = May, 2 = April, etc.).
* **$w(k)$**: The **weight** given to each month.

### Two Approaches We Will Use
If we estimated a separate weight for every month, we would have too many parameters (overfitting). We use two strategies to solve this:

1.  **Restricted MIDAS (Almon):** We force the weights to follow a smooth polynomial curve. Instead of estimating $K+1$ weights (e.g., 12 if we want to look back to a year of monthly data), we only estimate 3 or 4 parameters that describe the *shape* of the curve.
2.  **U-MIDAS (Unrestricted):** We simply treat the 3 months of the quarter as distinct variables ($m_1, m_2, m_3$). This works well when we don't need many lags.

In [ ]:
# Prepare Data
# We use the 'ext' series which now have data up to June 2023

# Calculate Growth/Diff for the high frequency (HF) Series
hf_consu = np.log(df_m['consu_ext']).diff()
hf_pmi = df_m['pmi_ext'].diff()

# Low Frequency (LF) controls from Quarterly Data
lf_cols = ['dlog_rgdp_1', 'dlog_rgdp_2', 'dlog_rgdp_3', 'dlog_us_1', 'dlog_cn_1', 'CRISIS']
df_lf = df_q[lf_cols].copy()
target = df_q['dlog_rgdp']

print("MIDAS Data Prepared.")

### 5.1 Almon MIDAS (Restricted)
We use an **Almon Polynomial** to constrain the coefficients of the monthly lags. This reduces the number of parameters to estimate (e.g., just 3 or 4 parameters describing the shape of the lag weights).

In [ ]:
# MIDAS regressions are not implemented in the Python statsmodel package (yet).
# However, with AI is is easy to code it up based on the theoretical description.
class AlmonMidas:
    """
    Custom class to implement MIDAS with Almon Polynomial Lags.
    """
    def __init__(self, degree=3, max_lag=12):
        self.degree = degree
        self.max_lag = max_lag
        self.model = None

    def _almon_transform(self, series):
        """
        Transforms a monthly series into polynomial weight variables (Z).

        Logic:
        Instead of estimating 12 weights (w0...w11), we approximate the weight
        curve using a polynomial: w(k) = c0 + c1*k + c2*k^2 ...

        This allows us to create 'Z' variables that are linear combinations
        of the lags, reducing the regression to finding c0, c1, c2.
        """
        # 1. Create Lag Matrix (columns = lag 0 to max_lag)
        X_lags = []
        for lag in range(self.max_lag + 1):
            # Shift monthly series, then take the last value of the quarter
            lagged = series.shift(lag).resample('QE').last()
            X_lags.append(lagged)
        X_matrix = pd.concat(X_lags, axis=1)
        # 2. Apply Almon Polynomial Transformation
        Z_matrix = pd.DataFrame(index=X_matrix.index)
        for j in range(self.degree + 1):
            # Weights based on polynomial degree (k^j)
            weights = np.array([k**j for k in range(self.max_lag + 1)])
            Z_matrix[f'poly_{j}'] = X_matrix.dot(weights)
        return Z_matrix

    def fit(self, y, x_high_freqs, x_low_freqs):
        # Create Design Matrix
        Z_combined = pd.DataFrame(index=y.index)
        for name, series in x_high_freqs.items():
            Z = self._almon_transform(series)
            Z.columns = [f'{name}_Z{i}' for i in range(len(Z.columns))]
            Z_combined = Z_combined.join(Z, how='left')
        Z_combined = Z_combined.join(x_low_freqs, how='left')
        Z_combined = sm.add_constant(Z_combined)
        # Align and Fit OLS
        data = Z_combined.join(y.rename('Y'), how='inner').dropna()
        self.model = sm.OLS(data['Y'], data.drop(columns=['Y'])).fit()
        return self.model.summary()

    def predict(self, x_high_freqs, x_low_freqs):
        # Transform inputs exactly like fit()
        Z_combined = pd.DataFrame()
        for name, series in x_high_freqs.items():
            Z = self._almon_transform(series)
            Z.columns = [f'{name}_Z{i}' for i in range(len(Z.columns))]
            if Z_combined.empty: Z_combined = Z
            else: Z_combined = Z_combined.join(Z, how='outer')
        # Fill NaNs (if any) with 0 to allow prediction on full range
        Z_combined = Z_combined.join(x_low_freqs, how='left')
        Z_combined = sm.add_constant(Z_combined).fillna(0)
        return self.model.predict(Z_combined)

In [ ]:
# --- ALMON MIDAS: AUTO LAG SELECTION ---
mask_est = (target.index >= '2001-03-31') & (target.index <= '2023-03-31')

best_aic = float('inf')
best_lag = 12
midas_almon = None

print("Finding best MIDAS lag...")
for lag in range(3, 13):
    m = AlmonMidas(degree=3, max_lag=lag)
    m.fit(target[mask_est], {'CONSU': hf_consu, 'PMI': hf_pmi}, df_lf[mask_est])
    if m.model.aic < best_aic:
        best_aic = m.model.aic
        best_lag = lag
        midas_almon = m

print(f"Best Lag: {best_lag}")
print(midas_almon.model.summary())

# Nowcast
pred_almon = midas_almon.predict({'CONSU': hf_consu, 'PMI': hf_pmi}, df_lf)
val_almon = pred_almon.loc['2023-06-30']

# Level Calculation
last_actual_level = df_q.loc['2023-03-31', 'rgdp']
level_almon = last_actual_level * np.exp(val_almon)

print(f"\nALMON MIDAS NOWCAST 2023Q2:")
print(f"Growth: {val_almon:.4f} (Log Diff)")
print(f"Level:  {level_almon:.2f} (INR Billions)")

# --- PLOT IN-SAMPLE FIT (LEVELS) ---
# Predict over evaluation sample (2018-2023)
fitted_growth_almon = pred_almon.loc[eval_mask]

# Reconstruct Levels (Level_t = Level_{t-1} * exp(Growth_t))
# 'prev_levels' was defined in the Bridge section
fitted_levels_almon = prev_levels * np.exp(fitted_growth_almon)

plt.figure(figsize=(10, 5))
plt.plot(eval_data.index, eval_data['rgdp'], label='Actual', marker='o')
plt.plot(eval_data.index, fitted_levels_almon, label='Almon Fitted', linestyle='--')
plt.plot(pd.Timestamp('2023-06-30'), level_almon, label='Nowcast', marker='*', markersize=15)
plt.title('Almon MIDAS: Actual vs Fitted Levels')
plt.legend()
plt.show()

### 5.2 U-MIDAS (Unrestricted)
U-MIDAS assumes the number of lags is not too large compared to the sample size. Instead of polynomials, we just treat each monthly lag as a separate variable with its coefficent to be estimated. Here we will use the 3 months of the quarter ($m_1, m_2, m_3$) as three separate variables.

In [ ]:
# --- U-MIDAS ---
def create_skip_sampled(series, prefix):
    """
    Creates skip-sampled quarterly variables from a monthly series.
    Aligns m1, m2, m3 indices to the same Quarter so they form one row.
    """
    # 1. Group by Quarter
    groups = series.groupby(series.index.to_period('Q'))

    # 2. Extract 1st, 2nd, 3rd months
    m1 = groups.nth(0)
    m2 = groups.nth(1)
    m3 = groups.nth(2)

    # 3. Reindex to Period('Q') for alignment
    # If we don't do this, concat tries to align Jan 1 with Feb 1 and fails to stack them.
    m1.index = m1.index.to_period('Q')
    m2.index = m2.index.to_period('Q')
    m3.index = m3.index.to_period('Q')

    # 4. Concatenate (Aligns on Quarter)
    df = pd.concat([m1, m2, m3], axis=1)

    # 5. Convert Index back to Timestamp (Quarter End) to match target data
    df.index = df.index.to_timestamp(how='end').normalize()
    df.columns = [f'{prefix}_m1', f'{prefix}_m2', f'{prefix}_m3']

    return df

# Create U-MIDAS Variables
skip_consu = create_skip_sampled(hf_consu, 'consu')
skip_pmi = create_skip_sampled(hf_pmi, 'pmi')

# Prepare DataFrame
# We join the skip-sampled monthly data with the quarterly control variables
df_umidas = df_q[lf_cols].join([skip_consu, skip_pmi])

# Estimate U-MIDAS
formula_u = "dlog_rgdp ~ dlog_rgdp_1 + dlog_rgdp_2 + dlog_rgdp_3 + dlog_us_1 + dlog_cn_1 + CRISIS + consu_m1 + consu_m2 + consu_m3 + pmi_m1 + pmi_m2 + pmi_m3"

# Join target and dropna to ensure we only use complete quarters
est_data = df_umidas.join(target).loc[mask_est].dropna()

model_umidas = smf.ols(formula_u, data=est_data).fit()
print("U-MIDAS Equation - Estimation Results")
print(model_umidas.summary())

# Nowcast
target_u = df_umidas.loc['2023-06-30': '2023-06-30'].fillna(0)
val_umidas = model_umidas.predict(target_u).values[0]

# Level Calculation
level_umidas = last_actual_level * np.exp(val_umidas)

print(f"\nU-MIDAS NOWCAST 2023Q2:")
print(f"Growth: {val_umidas:.4f} (Log Diff)")
print(f"Level:  {level_umidas:.2f} (INR Billions)")

# --- PLOT IN-SAMPLE FIT (LEVELS) ---
# Predict over evaluation sample
pred_data_u = df_umidas.loc[eval_mask].fillna(0)
fitted_growth_umidas = model_umidas.predict(pred_data_u)

fitted_levels_umidas = prev_levels * np.exp(fitted_growth_umidas)

plt.figure(figsize=(10, 5))
plt.plot(eval_data.index, eval_data['rgdp'], label='Actual', marker='o')
plt.plot(eval_data.index, fitted_levels_umidas, label='U-MIDAS Fitted', linestyle='--')
plt.plot(pd.Timestamp('2023-06-30'), level_umidas, label='Nowcast', marker='*', markersize=15)
plt.title('U-MIDAS: Actual vs Fitted Levels')
plt.legend()
plt.show()

## 6. Model Validation (2018–2023)

Now that we have built three different nowcasting models (Bridge, Almon MIDAS, and U-MIDAS), it is time to evaluate their performance.

### The "Pseudo-Out-of-Sample" Test


We perform a historical validation:
1.  We travel back in time to **2018Q1**.
2.  We pretend we are in that quarter and ask each model to forecast GDP using *only* the data up to that point (monthly indicators for that quarter + lagged GDP).
3.  We repeat this for every quarter up to **2023Q1**.
4.  We compare the model forecasts against the **Actual GDP** to calculate error metrics (e.g., RMSE).

**Important caveat**

This is **not** a pseudo-real-time exercise for several reasons. The most important is that in real life we never have the full range of monthly data at the end of a quarter. A more appropriate evaluation would take into account the timing of data releases and forecast the missing monthly indicators as we did for nowcasting 2023Q2 GDP. Another important factor that we swept under the rug is that data are often heavily revised over time. If we did travel back in time to 2018Q1, we would find that the historical data looked different from what we see today.

In [ ]:
# =========================================================
# 6. MODEL VALIDATION: HISTORICAL EVALUATION (2018-2023)
# =========================================================
# We assume the models (model_bridge, midas_almon, model_umidas) are already fitted.
# We now travel back to 2018 and check how they would have performed.

# 1. Define Evaluation Period
eval_mask = (df_q.index >= '2018-03-31') & (df_q.index <= '2023-03-31')

# Get Actuals
actual_growth = df_q.loc[eval_mask, 'dlog_rgdp']
actual_levels = df_q.loc[eval_mask, 'rgdp']
lagged_levels = df_q['rgdp'].shift(1).loc[eval_mask] # Needed to convert growth -> levels

# 2. Generate Static Forecasts for Each Model
# BRIDGE
pred_bridge = model_bridge.predict(df_q.loc[eval_mask])

# ALMON MIDAS (Requires specific input dict)
pred_almon = midas_almon.predict({'CONSU': hf_consu, 'PMI': hf_pmi}, df_lf).loc[eval_mask]

# U-MIDAS (FillNA ensures no gaps break the prediction)
pred_umidas = model_umidas.predict(df_umidas.loc[eval_mask].fillna(0))

# ENSEMBLE (Simple Average of the 3)
pred_avg = (pred_bridge + pred_almon + pred_umidas) / 3

# 3. Consolidate into DataFrame
results = pd.DataFrame({
    'Actual': actual_growth,
    'Bridge': pred_bridge,
    'Almon-MIDAS': pred_almon,
    'U-MIDAS': pred_umidas,
    'Ensemble': pred_avg
})

# 4. Helper Function: EViews-Style Metrics
def get_metrics(actual, forecast):
    err = forecast - actual
    mse = (err**2).mean()
    rmse = np.sqrt(mse)
    mae = err.abs().mean()
    # Theil U
    u_stat = rmse / (np.sqrt((actual**2).mean()) + np.sqrt((forecast**2).mean()))
    return pd.Series({'RMSE': rmse, 'MAE': mae, 'Theil U': u_stat})

# Calculate Metrics for all models
metrics_df = results.drop(columns='Actual').apply(lambda x: get_metrics(results['Actual'], x))

# --- VISUALIZATION & REPORT ---

# A. Plot Growth Rates
plt.figure(figsize=(12, 5))
plt.plot(results.index, results['Actual'], 'k-o', linewidth=2, label='Actual')
plt.plot(results.index, results['Bridge'], '--', label='Bridge')
plt.plot(results.index, results['Almon-MIDAS'], '-.', label='Almon MIDAS')
plt.plot(results.index, results['U-MIDAS'], ':', label='U-MIDAS')
plt.title('In-Sample Validation: Growth Rates (2018-2023)')
plt.ylabel('Log Difference')
plt.legend()
plt.show()

# B. Print Evaluation Table
print("\nFORECAST EVALUATION STATISTICS (2018Q1 - 2023Q1)")
print("-" * 60)
print(metrics_df.T) # Transpose for readability
print("-" * 60)
print(f"Best Model (Lowest RMSE): {metrics_df.loc['RMSE'].idxmin()}")

## 7. Summary of Nowcasts for 2023Q2
Comparison of the Nowcasts for 2023Q2 produced by the three methods.

In [ ]:
results = pd.DataFrame({
    'Model': ['Bridge', 'MIDAS (Almon)', 'U-MIDAS'],
    'Growth (Log Diff)': [nowcast_growth, val_almon, val_umidas],
    'Level (INR Billions)': [nowcast_level, level_almon, level_umidas]
})

print(results)

# Plot Comparison (Levels)
results.set_index('Model')['Level (INR Billions)'].plot(
    kind='bar',
    title='Comparison of Nowcasts for 2023Q2 (Levels)',
    color=['skyblue', 'salmon', 'lightgreen'],
    ylim=(results['Level (INR Billions)'].min() * 0.99, results['Level (INR Billions)'].max() * 1.01)
)
plt.xticks(rotation=0)
plt.ylabel('Real GDP (INR Billions)')
plt.show()

# Plot Comparison (Growth)
results.set_index('Model')['Growth (Log Diff)'].plot(
    kind='bar',
    title='Comparison of Nowcasts for 2023Q2 (Growth)',
    color=['skyblue', 'salmon', 'lightgreen'],
    #ylim=(results['Growth (Log Diff)'].min() * 0.99, results['Growth (Log Diff)'].max() * 1.01)
)
plt.xticks(rotation=0)
plt.ylabel('Real GDP Growth (QoQ)')
plt.show()

# XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

In [16]:

%ls

 NWC_W1_notebook.ipynb  'W-1 Standard Nowcasting (Bridge, MIDAS, U-MIDAS)'/


In [3]:
# Push changes to Github
def git_sync(message="Auto-update from Colab"):
    """Adds, commits, and pushes all changes in one step."""
    # Ensure you are in the correct directory
    repo_path = f"/content/drive/MyDrive/Colab/{repo_name}"
    os.chdir(repo_path)

    # Run Git sequence
    !git add .
    !git commit -m "{message}"
    !git push origin main
    print(f"🚀 Synced to GitHub: {message}")

# To sync, just run this:
git_sync()


fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
🚀 Synced to GitHub: Update from Colab session
